# Lab dl_05 · Train a small GPT on TinyShakespeare (GPU)

**Goal.** Assemble a GPT-2-style decoder-only transformer, check its parameter count against the closed form, train it on
~1 MB of Shakespeare on your GPU, and generate text.

### Theory recap
A (pre-LN) block: $\;x \leftarrow x + \mathrm{Attn}(\mathrm{LN}_1(x)),\;\; x \leftarrow x + \mathrm{MLP}(\mathrm{LN}_2(x))$, with
$\mathrm{MLP}(u) = W_2\,\mathrm{GELU}(W_1u + b_1) + b_2$, hidden size $4d$. The model:
$$\text{logits} = \mathrm{LN}_f\big(\text{blocks}(E_{tok}[x] + E_{pos}[0..T-1])\big)\,E_{tok}^\top \quad\text{(weight tying)}.$$
Exact parameter count (biases everywhere, LayerNorm gains+biases, tied head):
$$N = Vd + Td + n_{layer}(12d^2 + 13d) + 2d.$$
Training minimises next-token cross-entropy; at initialisation the loss should be $\approx \ln V$ (uniform predictions).
Training compute $\approx 6ND$ FLOPs for $D$ tokens.

Credits: architecture and training recipe follow [nanoGPT](https://github.com/karpathy/nanoGPT) /
[build-nanogpt](https://github.com/karpathy/build-nanogpt) and GPT-2; data: TinyShakespeare from
[char-rnn](https://github.com/karpathy/char-rnn) (already downloaded to `data/tinyshakespeare.txt`).

**How this works.** Fill in every cell marked `# TODO` (they currently `raise NotImplementedError`). Run the notebook top to bottom with the working directory set to `labs/`. The last cell runs hidden checks on your functions and records the result in the Learning Center. Do the paper homework for the unlocking skill first — the lab assumes you can do the math by hand.

*This lab is original course material, inspired by Andrej Karpathy's open-source teaching code ([micrograd](https://github.com/karpathy/micrograd), [nn-zero-to-hero](https://github.com/karpathy/nn-zero-to-hero), [nanoGPT](https://github.com/karpathy/nanoGPT), [build-nanogpt](https://github.com/karpathy/build-nanogpt), [minbpe](https://github.com/karpathy/minbpe)). Watching the corresponding lecture after finishing is highly recommended.*

In [ ]:
import math
import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
torch.backends.cuda.matmul.allow_tf32 = True

## 1. Data: characters as tokens
`build_char_vocab(text)` → `(stoi, itos)` with ids in sorted character order. `get_batch(data, block_size, batch_size, generator=None, device='cpu')` samples random windows: `x = data[i:i+T]`, `y = data[i+1:i+T+1]`.

In [ ]:
def load_text(path="data/tinyshakespeare.txt"):
    return Path(path).read_text(encoding="utf-8")


def build_char_vocab(text):
    # TODO
    raise NotImplementedError


def encode(s, stoi):
    return [stoi[c] for c in s]


def decode(ids, itos):
    return "".join(itos[int(i)] for i in ids)


def get_batch(data, block_size, batch_size, generator=None, device="cpu"):
    # TODO
    raise NotImplementedError

In [ ]:
text = load_text()
stoi, itos = build_char_vocab(text)
data = torch.tensor(encode(text, stoi), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(len(text), "chars, vocab", len(stoi), repr("".join(stoi)))

## 2. The model
Use these names (the checks and lab dl_08 rely on them): `GPTConfig` dataclass; `CausalSelfAttention` with `c_attn` (d→3d) and `c_proj` (d→d); `MLP` with `c_fc` (d→4d), `c_proj` (4d→d), GELU; `Block` with `ln_1, attn, ln_2, mlp`; `GPT` with `wte, wpe, drop, blocks (ModuleList), ln_f, lm_head` (no bias, **tied** to `wte`). Initialise Linear/Embedding weights $\mathcal N(0, 0.02^2)$, biases 0, and residual projections (`c_proj.weight`) with std $0.02/\sqrt{2n_{layer}}$. You may use `F.scaled_dot_product_attention(..., is_causal=True)` now that you built it yourself in dl_04.

`GPT.forward(idx, targets=None)` returns `(logits (B,T,V), loss or None)`. `GPT.generate(idx, max_new_tokens, temperature=1.0, top_k=None, generator=None)` crops the context to `block_size` and appends sampled tokens.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 65
    block_size: int = 128
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    dropout: float = 0.0


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO (pre-LN residual)
        raise NotImplementedError


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        # TODO: modules, weight tying, init
        raise NotImplementedError

    def forward(self, idx, targets=None):
        # TODO
        raise NotImplementedError

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, generator=None):
        # TODO
        raise NotImplementedError


def count_params(model):
    return sum(p.numel() for p in model.parameters())   # PyTorch counts a tied tensor once


def gpt_param_formula(cfg):
    # TODO: closed form from the theory recap
    raise NotImplementedError

In [ ]:
cfg = GPTConfig(vocab_size=len(stoi), block_size=128, n_layer=4, n_head=4, n_embd=256, dropout=0.0)
model = GPT(cfg).to(DEVICE)
print("params:", count_params(model), " formula:", gpt_param_formula(cfg))
x, y = get_batch(train_data, cfg.block_size, 8, device=DEVICE)
print("initial loss", model(x, y)[1].item(), " ln V =", math.log(cfg.vocab_size))

## 3. Training on the GPU
AdamW ($\beta=(0.9, 0.95)$, weight decay 0.1), gradient clipping at 1.0, bf16 autocast on CUDA. With the config above, 2000 iterations of batch 64 take roughly a minute on an RTX 5070 Ti and reach a validation loss around 1.5–1.6 nats/char. Try the larger config afterwards (6 layers, 6 heads, 384 dims, block 256, dropout 0.2: ~10 M params, ~1.47 val loss after 5000 iters).

In [ ]:
@torch.no_grad()
def estimate_loss(model, data, block_size, batch_size=32, iters=20, device=DEVICE, seed=0):
    g = torch.Generator().manual_seed(seed)
    model.eval()
    losses = [model(*get_batch(data, block_size, batch_size, g, device))[1].item() for _ in range(iters)]
    model.train()
    return sum(losses) / len(losses)


def train_gpt(model, data, block_size, batch_size=32, max_iters=200, lr=3e-4, device=DEVICE, seed=0, log_every=0):
    # TODO: AdamW loop with clipping and autocast; return the list of training losses
    raise NotImplementedError

In [ ]:
t0 = time.time()
losses = train_gpt(model, train_data, cfg.block_size, batch_size=64, max_iters=2000, lr=1e-3, log_every=250)
if DEVICE == "cuda": torch.cuda.synchronize()
dt = time.time() - t0
tokens = 2000 * 64 * cfg.block_size
print(f"{dt:.1f}s, {tokens / dt:,.0f} tok/s, ~{6 * count_params(model) * tokens / dt / 1e12:.1f} TFLOP/s (6ND estimate)")
print("val loss:", estimate_loss(model, val_data, cfg.block_size))
plt.plot(losses); plt.ylabel("train loss"); plt.show()
torch.save({"cfg": cfg.__dict__, "model": model.state_dict(), "stoi": stoi}, "data/dl05_gpt.pt")  # reused in dl_08

In [ ]:
start = torch.tensor([encode("\n", stoi)], device=DEVICE)
for T in (0.5, 1.0):
    print(f"--- temperature {T} ---")
    print(decode(model.generate(start, 400, temperature=T, generator=torch.Generator(device=DEVICE).manual_seed(0))[0].tolist(), itos))

### Reflect (on paper)
1. Derive $12d^2 + 13d$ per block and check your `gpt_param_formula` by hand for the config above.
2. What fraction of the parameters is in the embedding table here? For GPT-2 small ($V=50257$, $d=768$)?
3. Your measured TFLOP/s vs. the GPU's peak: what is your MFU? What limits it at this tiny model size?

## Grade
Runs the hidden checks (fast, small inputs, fixed seeds) and reports to the local app.

In [ ]:
from lc_client import grade
grade("dl_05_gpt", globals())